In [1]:
import pandas as pd
import pymysql
import gspread 
from oauth2client.service_account import ServiceAccountCredentials
from datetime import datetime, timedelta
import os
import glob
from datetime import datetime
from clickhouse_driver import Client


In [6]:
full_calls = pd.DataFrame()
path_to_sql_calls = 'C:/Users/Guest/Desktop/недозвоны/Накопительный по недозвонам Май 2023'
all_files = len(os.listdir(path_to_sql_calls))
print(f'Всего файлов {all_files}')
n = 0

for i in range(n,all_files):
    files = sorted(glob.glob(path_to_sql_calls + "/*.csv"),reverse=True)
    full_calls = pd.DataFrame()
    print(f'Текущий файл # {n+1}')
    print(files[n])
        #Get file creation time
    #timestamp = os.path.getmtime(files[n])
    #file_date = datetime.fromtimestamp(timestamp)
    #yesterday = datetime.now() - timedelta(days=1)
        # Only continue if file was created yesterday
    #if file_date.date() != yesterday.date():
        #print('Файл не был создан вчера, пропускаем')
         #continue
    

    print('Читаем файл')
    calls = pd.read_csv(files[n], encoding='utf-8', sep=',', error_bad_lines=False)
    full_calls = full_calls.append(calls)
    print('Редактируем формат')
    
    full_calls['custom_queue_c']=''
    full_calls['category_calls']=''
    
    full_calls[['project', 'calldate', 'network_provider', 'count_good_calls_c', 'База', 'last_queue_c',
                'custom_queue_c', 'marker_c', 'town_c', 'city_c', 'category_calls', 'category', 'stop_auto',
                'Разговоры',
                'Звонки', 'Переводы', 'Заявки']] = full_calls[['project', 'calldate',
                                                               'network_provider', 'count_good_calls_c', 'База',
                                                               'last_queue_c',
                                                               'custom_queue_c', 'marker_c', 'town_c', 'city_c',
                                                               'category_calls', 'category',
                                                               'stop_auto', 'Разговоры', 'Звонки', 'Переводы',
                                                               'Заявки']].astype('str').fillna('')
    full_calls['calldate'] = pd.to_datetime(calls['calldate'])
    full_calls['marker_c'] = full_calls['marker_c'].astype(str)
    full_calls['marker_c'] = full_calls['marker_c'].apply(lambda x: x.replace('.0', ''))
    full_calls['category'] = full_calls['category'].astype(str)
    full_calls['category'] = full_calls['category'].apply(lambda x: x.replace('.0', ''))
    full_calls['city_c'] = full_calls['city_c'].astype(str)
    full_calls['city_c'] = full_calls['city_c'].apply(lambda x: x.replace('.0', ''))
    full_calls['stop_auto'] = full_calls['stop_auto'].astype(str)
    full_calls['stop_auto'] = full_calls['stop_auto'].apply(lambda x: x.replace('.0', ''))
    full_calls[['project', 'calldate', 'network_provider', 'count_good_calls_c', 'База', 'last_queue_c',
                'custom_queue_c', 'marker_c', 'town_c', 'city_c', 'category_calls', 'category', 'stop_auto',
                'Разговоры',
                'Звонки', 'Переводы', 'Заявки']] = full_calls[['project', 'calldate',
                                                               'network_provider', 'count_good_calls_c', 'База',
                                                               'last_queue_c',
                                                               'custom_queue_c', 'marker_c', 'town_c', 'city_c',
                                                               'category_calls', 'category',
                                                               'stop_auto', 'Разговоры', 'Звонки', 'Переводы',
                                                               'Заявки']].astype('str').fillna('')
    full_calls = full_calls.rename(columns={'База': 'base',
                                            'Разговоры': 'talks', 'Звонки': 'calls', 'Переводы': 'perevod',
                                            'Заявки': 'meeting'}).fillna('')

    full_calls['talks'] = full_calls['talks'].astype('int64')
    full_calls['calls'] = full_calls['calls'].astype('int64')
    full_calls['perevod'] = full_calls['perevod'].astype('int64')
    full_calls['meeting'] = full_calls['meeting'].astype('int64')
    full_calls = full_calls.groupby(['project', 
                            'calldate', 
                            'network_provider',
                            'count_good_calls_c',
                            'base', 'last_queue_c',
                            'custom_queue_c', 'marker_c', 
                            'town_c', 'city_c', 
                            'category_calls', 'category', 
                            'stop_auto'],as_index=False, dropna=False).agg({'talks': 'sum',
                                                    'calls': 'sum',
                                                    'perevod': 'sum',
                                                    'meeting': 'sum'})
    
    
    full_calls[['talks','calls','perevod','meeting']] = full_calls[['talks','calls','perevod','meeting']].astype('int64').fillna(0)


    print('Загрузка в базу')
    client = Client(host='192.168.1.99', port='9000', user='default', password='jdfwl6812hwe',
                    database='suitecrm_robot_ch', settings={'use_numpy': True})

    client.insert_dataframe('INSERT INTO suitecrm_robot_ch.nakopitelny_nedozvons VALUES', full_calls)
    n += 1
    if n == 49:
        break

Всего файлов 31
Текущий файл # 1
C:/Users/Guest/Desktop/недозвоны/Накопительный по недозвонам Май 2023\Дозвон_05_31.csv
Читаем файл


C:\Users\Guest\AppData\Local\Temp\ipykernel_7364\2475661276.py:23: FutureWarning: The error_bad_lines argument has been deprecated and will be removed in a future version. Use on_bad_lines in the future.


  calls = pd.read_csv(files[n], encoding='utf-8', sep=',', error_bad_lines=False)
C:\Users\Guest\AppData\Local\Temp\ipykernel_7364\2475661276.py:24: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  full_calls = full_calls.append(calls)


Редактируем формат
Загрузка в базу
Текущий файл # 2
C:/Users/Guest/Desktop/недозвоны/Накопительный по недозвонам Май 2023\Дозвон_05_30.csv
Читаем файл


C:\Users\Guest\AppData\Local\Temp\ipykernel_7364\2475661276.py:23: FutureWarning: The error_bad_lines argument has been deprecated and will be removed in a future version. Use on_bad_lines in the future.


  calls = pd.read_csv(files[n], encoding='utf-8', sep=',', error_bad_lines=False)
C:\Users\Guest\AppData\Local\Temp\ipykernel_7364\2475661276.py:23: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  calls = pd.read_csv(files[n], encoding='utf-8', sep=',', error_bad_lines=False)
C:\Users\Guest\AppData\Local\Temp\ipykernel_7364\2475661276.py:24: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  full_calls = full_calls.append(calls)


Редактируем формат
Загрузка в базу
Текущий файл # 3
C:/Users/Guest/Desktop/недозвоны/Накопительный по недозвонам Май 2023\Дозвон_05_29.csv
Читаем файл


...

Редактируем формат
Загрузка в базу
Текущий файл # 30
C:/Users/Guest/Desktop/недозвоны/Накопительный по недозвонам Май 2023\Дозвон_05_02.csv
Читаем файл


C:\Users\Guest\AppData\Local\Temp\ipykernel_7364\2475661276.py:23: FutureWarning: The error_bad_lines argument has been deprecated and will be removed in a future version. Use on_bad_lines in the future.


  calls = pd.read_csv(files[n], encoding='utf-8', sep=',', error_bad_lines=False)
C:\Users\Guest\AppData\Local\Temp\ipykernel_7364\2475661276.py:24: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  full_calls = full_calls.append(calls)


Редактируем формат
Загрузка в базу
Текущий файл # 31
C:/Users/Guest/Desktop/недозвоны/Накопительный по недозвонам Май 2023\Дозвон_05_01.csv
Читаем файл


C:\Users\Guest\AppData\Local\Temp\ipykernel_7364\2475661276.py:23: FutureWarning: The error_bad_lines argument has been deprecated and will be removed in a future version. Use on_bad_lines in the future.


  calls = pd.read_csv(files[n], encoding='utf-8', sep=',', error_bad_lines=False)
C:\Users\Guest\AppData\Local\Temp\ipykernel_7364\2475661276.py:24: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  full_calls = full_calls.append(calls)


Редактируем формат
Загрузка в базу
